In [2]:
import pandas as pd

df = pd.read_csv("../data/clean/newcastle_raw_filtered.csv", parse_dates=["date"])
n0 = len(df)
log = []

df["year"] = df["date"].dt.year
df["postcode"] = df["postcode"].str.strip().str.upper()

# a) Deleted records
before = len(df)
df = df[df["record_status"] != "D"]
log.append(f"Removed deleted records (record_status = D): {before - len(df)} rows")

# b) Duplicate transactions
before = len(df)
df = df.drop_duplicates(subset="transaction_id")
log.append(f"Removed duplicate transaction IDs: {before - len(df)} rows")

# c) Missing postcodes
before = len(df)
df = df.dropna(subset=["postcode"])
log.append(f"Removed rows with missing postcode: {before - len(df)} rows")

# d) Extract postcode district (NE1, NE3, ...)
df["pc_district"] = df["postcode"].str.split(" ").str[0]

# e) Keep only the four main property types
before = len(df)
df = df[df["property_type"].isin(["D", "S", "T", "F"])]
log.append(f"Removed property_type = O (other): {before - len(df)} rows")

# f) Implausibly low prices
before = len(df)
df = df[df["price"] >= 10_000]
log.append(f"Removed prices below GBP 10,000: {before - len(df)} rows")

# g) Price outliers (IQR x 3, per property type)
q1 = df.groupby("property_type")["price"].transform(lambda s: s.quantile(0.25))
q3 = df.groupby("property_type")["price"].transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
before = len(df)
df = df[(df["price"] >= q1 - 3 * iqr) & (df["price"] <= q3 + 3 * iqr)]
log.append(f"Removed price outliers (3x IQR per property type): {before - len(df)} rows")

log.append(f"START: {n0} rows, END: {len(df)} rows")
print("\n".join(log))

Removed deleted records (record_status = D): 0 rows
Removed duplicate transaction IDs: 0 rows
Removed rows with missing postcode: 63 rows
Removed property_type = O (other): 2248 rows
Removed prices below GBP 10,000: 1 rows
Removed price outliers (3x IQR per property type): 851 rows
START: 39916 rows, END: 36753 rows


In [3]:
# Readable labels
df["property_type"] = df["property_type"].map(
    {"D": "Detached", "S": "Semi-detached", "T": "Terraced", "F": "Flat"})
df["tenure"] = df["tenure"].map({"F": "Freehold", "L": "Leasehold"})
df["new_build"] = df["new_build"].map({"Y": True, "N": False})

# Sanity checks
print("Top 15 postcode districts by number of sales:")
print(df["pc_district"].value_counts().head(15))
print("\nMedian price by year:")
print(df.groupby("year")["price"].median())

# Save clean dataset
final = df[["transaction_id", "price", "date", "year", "postcode", "pc_district",
            "property_type", "new_build", "tenure", "street", "locality"]]
final.to_csv("../data/clean/newcastle_clean.csv", index=False)

# Save cleaning log
with open("../cleaning_log.md", "w", encoding="utf-8") as f:
    f.write("# Cleaning Log\n\n" + "\n".join(f"- {l}" for l in log))
print("\nSaved: newcastle_clean.csv and cleaning_log.md")

Top 15 postcode districts by number of sales:
pc_district
NE5     6976
NE3     6849
NE6     5052
NE13    4853
NE15    4569
NE2     2728
NE4     2461
NE7     1939
NE1     1299
NE16      10
NE27       4
NE12       3
SR2        2
NE17       1
NE21       1
Name: count, dtype: int64

Median price by year:
year
2018    155000.0
2019    157247.5
2020    165000.0
2021    175000.0
2022    185000.0
2023    185000.0
2024    187500.0
2025    190500.0
2026    190500.0
Name: price, dtype: float64

Saved: newcastle_clean.csv and cleaning_log.md
